In [1]:
from PIL import Image # We use the PIL Library to resize images
import numpy as np
import os
import cv2
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Dense,Flatten,Dropout
data = []
labels = []

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
ls

drive/  sample_data/


In [4]:
cd /content/drive/MyDrive/project_skin/

/content/drive/MyDrive/project_skin


In [5]:
ls

'skin diseases.zip'   test/   train/


In [6]:
ls

'skin diseases.zip'   test/   train/


In [ ]:
!unzip skin\ diseases.zip

Archive:  skin diseases.zip
replace train/acne/ac100_JPG.rf.a41d4a786f2197b20cfa6372cc1625a5.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [7]:
ls

'skin diseases.zip'   test/   train/


In [8]:
src_path_train = r'train'


In [9]:
src_path_test= r'test'

In [10]:
batch_size = 8

In [11]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1 / 255.0,
        rotation_range=20,
        zoom_range=0.05,
        width_shift_range=0.05,
        height_shift_range=0.05,
        shear_range=0.05,
        horizontal_flip=True,
        fill_mode="nearest",
        validation_split=0.20)


In [12]:
train_generator = train_datagen.flow_from_directory(
    directory=src_path_train,
    target_size=(100, 100),
    color_mode="rgb",
    batch_size=batch_size,
    class_mode="categorical",
    subset='training',
    shuffle=True,
    seed=42
)

Found 1188 images belonging to 5 classes.


In [13]:
valid_generator = train_datagen.flow_from_directory(
    directory=src_path_train,
    target_size=(100, 100),
    color_mode="rgb",
    batch_size=batch_size,
    class_mode="categorical",
    subset='validation',
    shuffle=True,
    seed=42
)

Found 295 images belonging to 5 classes.


In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

In [16]:
test_generator = test_datagen.flow_from_directory(
    directory=src_path_test,
    target_size=(100, 100),
    color_mode="rgb",
    batch_size=1,
    class_mode=None,
    shuffle=False,
    seed=42
)

Found 11 images belonging to 5 classes.


In [17]:
train_generator.image_shape

(100, 100, 3)

In [18]:
def prepare_model():
    model = Sequential()
    model.add(Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(100, 100, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(16, activation='relu'))

    model.add(Dense(4, activation='softmax'))  # For multi-class classification
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=['accuracy'])
    return model

In [25]:
from tensorflow.keras.applications import VGG16 #Densenet Resnet efficientnet inception
#xception
#transformers---attention based networks
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
def prepare_model():
    # Load VGG16 with pre-trained weights from ImageNet and exclude the top layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(100, 100, 3))

    # Freeze the base model to retain pre-trained features during the initial training phase
    base_model.trainable = False

    # Add custom layers on top of the base VGG16 model
    model = Sequential()
    model.add(base_model)  # Add the VGG16 convolutional base

    # Add a global average pooling layer
    model.add(GlobalAveragePooling2D())

    # Add a fully connected layer
    model.add(Dense(16, activation='relu'))

    # Add the final output layer for multi-class classification (4 classes)
    model.add(Dense(5, activation='softmax'))

    # Compile the model
    model.compile(loss="categorical_crossentropy", optimizer=Adam(), metrics=['accuracy'])

    return model

In [26]:
model = prepare_model()

In [24]:
print(train_generator.class_indices)
print(len(train_generator.class_indices))

{'acne': 0, 'eksim': 1, 'herpes': 2, 'panu': 3, 'rosacea': 4}
5


In [27]:
model.fit(
    train_generator,
    validation_data=valid_generator,
    steps_per_epoch=train_generator.n // train_generator.batch_size,
    validation_steps=valid_generator.n // valid_generator.batch_size,
    epochs=5
)
#model.fit(train_generator,
                    #validation_data = train_generator,
                    #steps_per_epoch = train_generator.n//train_generator.batch_size,
                    #validation_steps = valid_generator.n//valid_generator.batch_size,
                    #epochs=5)

Epoch 1/5
 59/148 ━━━━━━━━━━━━━━━━━━━━ 3:29 2s/step - accuracy: 0.1904 - loss: 1.6084

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam

def prepare_model():
    # Load VGG16 with pre-trained weights from ImageNet and exclude the top layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(100, 100, 3))

    # Freeze the base model to retain pre-trained features during the initial training phase
    base_model.trainable = True

    # Fine-tune from a specific layer onward (e.g., unfreeze layers after the 15th layer)
    for layer in base_model.layers[:13]:
        layer.trainable = False

    # Recompile the model to apply the changes
    model.compile(loss="categorical_crossentropy", optimizer=Adam(learning_rate=1e-5), metrics=['accuracy'])

    return model

# Summary of the model
model = prepare_model()
model.summary()
model.fit(train_generator,
                    validation_data = train_generator,
                    steps_per_epoch = train_generator.n//train_generator.batch_size,
                    validation_steps = valid_generator.n//valid_generator.batch_size,
                    epochs=5)


In [ ]:
score = model.evaluate(valid_generator)
print('Test loss:', score[0])
print('Test accuracy:', score[1]*100)

In [ ]:
import cv2

while(True):
    img=cv2.VideoCapture(1)
    img.resize(100,100) ---50 50 model trained
    temp=object detection-->x y w h

    model.predict(temp)
    cv2.rectangle(x y w h)